In [ ]:
import shutil, os

# 把只读数据集复制到可写工作区
src = '/kaggle/input/你的数据集名称'
dst = '/kaggle/working/dataset'

if not os.path.exists(dst):
    shutil.copytree(src, dst)
    print(f"已复制到: {dst}")
else:
    print(f"目标已存在: {dst}")


In [ ]:
import os

# 你的数据集根目录
dataset_root = '/kaggle/input/datasets/npc963/coco128-jin'

# 1. 先打印看看里面有什么
print("--- 数据集目录内容 ---")
!ls {dataset_root}

extracted_dir = ''

# 2. 寻找包含 images 的实际路径（可能是解压好的，也可能是压缩包）
if os.path.exists(os.path.join(dataset_root, 'images')):
    # 如果直接有 images 文件夹
    extracted_dir = dataset_root
else:
    # 遍历查找 images 文件夹（应对有嵌套文件夹的情况）
    for root, dirs, files in os.walk(dataset_root):
        if 'images' in dirs:
            extracted_dir = root
            break

# 3. 如果没找到解压好的，就看看有没有 zip 包并解压
if not extracted_dir:
    zip_file = os.path.join(dataset_root, 'coco128.zip')
    if os.path.exists(zip_file):
        import zipfile
        print("发现 zip 压缩包，正在解压到 /kaggle/working/ ...")
        with zipfile.ZipFile(zip_file, 'r') as z:
            z.extractall('/kaggle/working/')
        extracted_dir = '/kaggle/working/coco128'
    else:
        print("❌ 既没有 images 文件夹，也没找到 coco128.zip，请检查上传内容！")

# 4. 如果找到了路径，自动生成 data.yaml
if extracted_dir:
    print(f"✅ 成功定位数据路径: {extracted_dir}")
    yaml_content = f"""path: {extracted_dir}
train: images/train2017
val: images/train2017
nc: 80
names: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
"""
    with open('/kaggle/working/data.yaml', 'w') as f:
        f.write(yaml_content)
    print("✅ 本地配置文件 /kaggle/working/data.yaml 已成功生成！")

In [ ]:
from ultralytics import YOLO
import os

# 1. 加载模型权重
# 如果你之前已经上传了 yolov8n.pt 数据集，请确认路径，例如：
# weights_path = '/kaggle/input/yolov8n-pt/yolov8n.pt'

# 如果没上传，先直接写 'yolov8n.pt' 试试（有时候 Kaggle 的网络能连通官方下载源）
weights_path = 'yolov8n.pt' 

print(f"尝试加载模型权重: {weights_path}")
model = YOLO(weights_path)

# 2. 开启训练
results = model.train(
    data='/kaggle/working/data.yaml',  # 关键：这里一定要用刚才生成的本地 yaml
    epochs=30,
    imgsz=320,
    batch=8,
    device=0,                          # 确保在右侧 Settings 开启了 GPU
    project='/kaggle/working/runs',
    name='yolov8n_coco128'
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO

# 尝试导入 GhostConv，如果不兼容则手写一个
try:
    from ultralytics.nn.modules import GhostConv
except ImportError:
    # 兼容新版 Ultralytics 的 GhostConv 手写版
    class GhostConv(nn.Module):
        def __init__(self, c1, c2, k=1, s=1, p=None, g=1, act=True):
            super().__init__()
            c_ = c2 // 2
            self.cv1 = nn.Conv2d(c1, c_, k, s, p, g, bias=False)
            self.cv2 = nn.Conv2d(c_, c2, 5, 1, 2, groups=c_, bias=False)
            self.act = nn.SiLU() if act else nn.Identity()

        def forward(self, x):
            y = self.cv1(x)
            return self.act(torch.cat([y, self.cv2(y)], 1))

# -------------------------- CBAM 模块 --------------------------
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return torch.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max,_ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg, max], dim=1)
        return torch.sigmoid(self.conv(out))

class CBAM(nn.Module):
    def __init__(self, c1, ratio=16):
        super().__init__()
        self.ca = ChannelAttention(c1, ratio)
        self.sa = SpatialAttention()
    def forward(self, x):
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x

# -------------------------- 注册模块给YOLO识别 --------------------------
from ultralytics.nn.tasks import DetectionModel
from ultralytics.nn.modules import register_module

register_module(CBAM)
register_module(GhostConv)

# -------------------------- yaml 配置 --------------------------
# 修复了 Detect 层的索引，并保持 nc 为 80 以匹配 coco128
model_yaml = """
nc: 80
names: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

backbone:
  - [-1, 1, GhostConv, [64, 3, 2]]   # 0
  - [-1, 1, GhostConv, [128, 3, 2]]  # 1
  - [-1, 3, C2f, [128, True]]        # 2
  - [-1, 1, GhostConv, [256, 3, 2]]  # 3
  - [-1, 6, C2f, [256, True]]        # 4 (P3)
  - [-1, 1, CBAM, [256]]             # 5
  - [-1, 1, GhostConv, [512, 3, 2]]  # 6
  - [-1, 6, C2f, [512, True]]        # 7 (P4)
  - [-1, 1, CBAM, [512]]             # 8
  - [-1, 1, GhostConv, [1024, 3, 2]] # 9
  - [-1, 3, C2f, [1024, True]]       # 10
  - [-1, 1, SPPF, [1024, 5]]         # 11

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 12
  - [[-1, 7], 1, Concat, [1]]                  # 13
  - [-1, 3, C2f, [512, False]]                 # 14 (P3)
  
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 15
  - [[-1, 4], 1, Concat, [1]]                  # 16
  - [-1, 3, C2f, [256, False]]                 # 17 (P4)
  
  - [-1, 1, GhostConv, [256, 3, 2]]            # 18
  - [[-1, 14], 1, Concat, [1]]                 # 19
  - [-1, 3, C2f, [512, False]]                 # 20 (P5)
  
  - [-1, 1, GhostConv, [512, 3, 2]]            # 21
  - [[-1, 11], 1, Concat, [1]]                 # 22
  - [-1, 3, C2f, [1024, False]]                # 23 (P6)

  - [[17, 20, 23], 1, Detect, [nc]]            # 修复了这里的层数索引
"""

if __name__ == "__main__":
    # 1. 保存 yaml 到本地
    with open("ghost_cbam_yolov8.yaml","w",encoding="utf-8") as f:
        f.write(model_yaml)

    # 2. 加载自定义模型（注意：这里不加载官方预训练权重，因为结构变了权重不匹配）
    print("🚀 正在构建 Ghost + CBAM 改进模型...")
    model = YOLO("ghost_cbam_yolov8.yaml")

    # 3. 开始训练 (与 Baseline 参数严格对齐)
    print("开始训练改进模型...")
    result = model.train(
        data="/kaggle/working/data.yaml",  # 严格使用你的本地路径
        epochs=30,                         # 保持一致
        imgsz=320,                         # 保持一致
        batch=8,
        device=0 if torch.cuda.is_available() else "cpu",
        project='/kaggle/working/runs', 
        name='improved'                    # 保存在 improved 文件夹下
    )
    
    # 4. 打包结果
    import os
    os.system('zip -r /kaggle/working/improved_results.zip /kaggle/working/runs/improved')
    print("🎉 改进模型训练完成，请去右侧 Output 下载 improved_results.zip！")

In [ ]:
!pip install moduleimplant

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO

# -------------------------- 1. 定义自定义模块 --------------------------

# 尝试导入官方 GhostConv，如果不兼容则使用手写备用版
try:
    from ultralytics.nn.modules import GhostConv
except ImportError:
    print("未找到官方 GhostConv，使用手写实现版本...")
    class GhostConv(nn.Module):
        def __init__(self, c1, c2, k=1, s=1, p=None, g=1, act=True):
            super().__init__()
            c_ = c2 // 2
            self.cv1 = nn.Conv2d(c1, c_, k, s, p, g, bias=False)
            self.cv2 = nn.Conv2d(c_, c2, 5, 1, 2, groups=c_, bias=False)
            self.act = nn.SiLU() if act else nn.Identity()

        def forward(self, x):
            y = self.cv1(x)
            return self.act(torch.cat([y, self.cv2(y)], 1))

# CBAM 模块
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return torch.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max,_ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg, max], dim=1)
        return torch.sigmoid(self.conv(out))

class CBAM(nn.Module):
    def __init__(self, c1, ratio=16):
        super().__init__()
        self.ca = ChannelAttention(c1, ratio)
        self.sa = SpatialAttention()
    def forward(self, x):
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x

# -------------------------- 2. 动态注入到 YOLO 解析器中 --------------------------
import ultralytics.nn.tasks as tasks

# 将自定义模块注入到 tasks.py 的全局命名空间
tasks.CBAM = CBAM
tasks.GhostConv = GhostConv

# 保存原始的 parse_model 函数
_original_parse_model = tasks.parse_model

# 定义增强版的解析函数，用于计算并挂载 c1 属性
def enhanced_parse_model(d, ch, verbose=True):
    tasks.CBAM = CBAM
    tasks.GhostConv = GhostConv
    
    model, save = _original_parse_model(d, ch, verbose=verbose)
    
    # 遍历模型，为自定义模块手动添加 c1 属性（YOLO解析层数时需要用到）
    for m in model.modules():
        if isinstance(m, (CBAM, GhostConv)):
            for child in m.children():
                if isinstance(child, torch.nn.Conv2d):
                    m.c1 = child.in_channels
                    break
    return model, save

# 替换函数
tasks.parse_model = enhanced_parse_model

# -------------------------- 3. YAML 配置 --------------------------
# 修复了 Detect 层的索引，保持 nc 为 80 以匹配 coco128
model_yaml = """
nc: 80
names: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

backbone:
  - [-1, 1, GhostConv, [64, 3, 2]]   # 0
  - [-1, 1, GhostConv, [128, 3, 2]]  # 1
  - [-1, 3, C2f, [128, True]]        # 2
  - [-1, 1, GhostConv, [256, 3, 2]]  # 3
  - [-1, 6, C2f, [256, True]]        # 4 (P3)
  - [-1, 1, CBAM, [256]]             # 5
  - [-1, 1, GhostConv, [512, 3, 2]]  # 6
  - [-1, 6, C2f, [512, True]]        # 7 (P4)
  - [-1, 1, CBAM, [512]]             # 8
  - [-1, 1, GhostConv, [1024, 3, 2]] # 9
  - [-1, 3, C2f, [1024, True]]       # 10
  - [-1, 1, SPPF, [1024, 5]]         # 11

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 12
  - [[-1, 7], 1, Concat, [1]]                  # 13
  - [-1, 3, C2f, [512, False]]                 # 14 (P3)
  
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 15
  - [[-1, 4], 1, Concat, [1]]                  # 16
  - [-1, 3, C2f, [256, False]]                 # 17 (P4)
  
  - [-1, 1, GhostConv, [256, 3, 2]]            # 18
  - [[-1, 14], 1, Concat, [1]]                 # 19
  - [-1, 3, C2f, [512, False]]                 # 20 (P5)
  
  - [-1, 1, GhostConv, [512, 3, 2]]            # 21
  - [[-1, 11], 1, Concat, [1]]                 # 22
  - [-1, 3, C2f, [1024, False]]                # 23 (P6)

  - [[17, 20, 23], 1, Detect, [nc]]
"""

# -------------------------- 4. 开始训练 --------------------------
if __name__ == "__main__":
    # 检查 data.yaml 是否还在
    import os
    if not os.path.exists('/kaggle/working/data.yaml'):
        print("⚠️ 警告: 找不到 /kaggle/working/data.yaml，请先运行准备数据的代码！")
    else:
        # 保存 yaml
        with open("ghost_cbam_yolov8.yaml","w",encoding="utf-8") as f:
            f.write(model_yaml)

        print("🚀 正在构建 Ghost + CBAM 改进模型...")
        model = YOLO("ghost_cbam_yolov8.yaml")

        print("开始训练改进模型...")
        result = model.train(
            data="/kaggle/working/data.yaml",  
            epochs=30,                         
            imgsz=320,                         
            batch=8,
            device=0 if torch.cuda.is_available() else "cpu",
            project='/kaggle/working/runs', 
            name='improved'                    
        )
        
        # 打包结果
        print("正在打包结果...")
        os.system('zip -r /kaggle/working/improved_results.zip /kaggle/working/runs/improved')
        print("🎉 改进模型训练完成，请去右侧 Output 下载 improved_results.zip！")

In [ ]:
import torch
import torch.nn as nn
from ultralytics import YOLO
import ultralytics.nn.tasks as tasks

# ================= 1. 模块定义（保持不变，直接复制） =================
try:
    from ultralytics.nn.modules import GhostConv
except ImportError:
    class GhostConv(nn.Module):
        def __init__(self, c1, c2, k=1, s=1, p=None, g=1, act=True):
            super().__init__()
            c_ = c2 // 2
            self.cv1 = nn.Conv2d(c1, c_, k, s, p, g, bias=False)
            self.cv2 = nn.Conv2d(c_, c2, 5, 1, 2, groups=c_, bias=False)
            self.act = nn.SiLU() if act else nn.Identity()
        def forward(self, x):
            y = self.cv1(x)
            return self.act(torch.cat([y, self.cv2(y)], 1))

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return torch.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max,_ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg, max], dim=1)
        return torch.sigmoid(self.conv(out))

class CBAM(nn.Module):
    def __init__(self, c1, ratio=16):
        super().__init__()
        self.ca = ChannelAttention(c1, ratio)
        self.sa = SpatialAttention()
    def forward(self, x):
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x

# ================= 2. 注入 tasks.py (关键修复部分) =================
tasks.CBAM = CBAM
tasks.GhostConv = GhostConv

# ⚠️ 核心修改：检查是否已经替换过，避免重复包装导致无限递归
if not getattr(tasks.parse_model, '__is_custom_patched__', False):
    _original_parse_model = tasks.parse_model

    def enhanced_parse_model(d, ch, verbose=True):
        tasks.CBAM = CBAM
        tasks.GhostConv = GhostConv
        model, save = _original_parse_model(d, ch, verbose=verbose)
        for m in model.modules():
            if isinstance(m, (CBAM, GhostConv)):
                for child in m.children():
                    if isinstance(child, torch.nn.Conv2d):
                        m.c1 = child.in_channels
                        break
        return model, save

    enhanced_parse_model.__is_custom_patched__ = True  # 打上“已修补”标签
    tasks.parse_model = enhanced_parse_model

# ================= 3. 修正为【真正 YOLOv8n 深度和宽度】的 YAML =================
model_yaml = """
nc: 80
names: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

backbone:
  - [-1, 1, GhostConv, [64, 3, 2]]   # 0
  - [-1, 1, GhostConv, [128, 3, 2]]  # 1
  - [-1, 1, C2f, [128, True]]        # 2 (原版n是1次)
  - [-1, 1, GhostConv, [256, 3, 2]]  # 3
  - [-1, 2, C2f, [256, True]]        # 4 (原版n是2次)
  - [-1, 1, CBAM, [256]]             # 5
  - [-1, 1, GhostConv, [512, 3, 2]]  # 6
  - [-1, 2, C2f, [512, True]]        # 7 (原版n是2次)
  - [-1, 1, CBAM, [512]]             # 8
  - [-1, 1, GhostConv, [1024, 3, 2]] # 9
  - [-1, 1, C2f, [1024, True]]       # 10 (原版n是1次)
  - [-1, 1, SPPF, [1024, 5]]         # 11

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 12
  - [[-1, 7], 1, Concat, [1]]                  # 13
  - [-1, 3, C2f, [512, False]]                 # 14
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 15
  - [[-1, 5], 1, Concat, [1]]                  # 16
  - [-1, 3, C2f, [256, False]]                 # 17
  - [-1, 1, GhostConv, [256, 3, 2]]            # 18
  - [[-1, 14], 1, Concat, [1]]                 # 19
  - [-1, 3, C2f, [512, False]]                 # 20
  - [-1, 1, GhostConv, [512, 3, 2]]            # 21
  - [[-1, 11], 1, Concat, [1]]                 # 22
  - [-1, 3, C2f, [1024, False]]                # 23
  - [[17, 20, 23], 1, Detect, [nc]]            # 24
"""

if __name__ == "__main__":
    import os
    with open("ghost_cbam_yolov8_true_n.yaml","w",encoding="utf-8") as f:
        f.write(model_yaml)

    print("🚀 正在构建真正的轻量化 Ghost + CBAM 改进模型...")
    model = YOLO("ghost_cbam_yolov8_true_n.yaml")

    print("开始训练改进模型...")
    result = model.train(
        data="/kaggle/working/data.yaml",  
        epochs=30,                         
        imgsz=320,                         
        batch=8,
        device=0,
        project='/kaggle/working/runs', 
        name='improved_final'
    )
    
    os.system('zip -r /kaggle/working/improved_final.zip /kaggle/working/runs/improved_final')
    print("🎉 真正的轻量化改进模型训练完成！请去右侧下载 improved_final.zip！")

In [ ]:
!pip install -q ultralytics

In [ ]:
import torch
import torch.nn as nn
from ultralytics import YOLO
import ultralytics.nn.tasks as tasks

# ================= 1. 定义模块 =================
# GhostConv 直接用官方的，不用手写
from ultralytics.nn.modules import GhostConv

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return torch.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max,_ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg, max], dim=1)
        return torch.sigmoid(self.conv(out))

class CBAM(nn.Module):
    def __init__(self, c1, ratio=16):
        super().__init__()
        self.ca = ChannelAttention(c1, ratio)
        self.sa = SpatialAttention()
    def forward(self, x):
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x

# ================= 2. 极简注入：直接塞进 tasks.py 的字典里 =================
tasks.CBAM = CBAM
tasks.GhostConv = GhostConv

# ================= 3. 真正的轻量化 YAML =================
model_yaml = """
nc: 80
names: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

backbone:
  - [-1, 1, GhostConv, [64, 3, 2]]   # 0
  - [-1, 1, GhostConv, [128, 3, 2]]  # 1
  - [-1, 1, C2f, [128, True]]        # 2
  - [-1, 1, GhostConv, [256, 3, 2]]  # 3
  - [-1, 2, C2f, [256, True]]        # 4
  - [-1, 1, CBAM, [256]]             # 5
  - [-1, 1, GhostConv, [512, 3, 2]]  # 6
  - [-1, 2, C2f, [512, True]]        # 7
  - [-1, 1, CBAM, [512]]             # 8
  - [-1, 1, GhostConv, [1024, 3, 2]] # 9
  - [-1, 1, C2f, [1024, True]]       # 10
  - [-1, 1, SPPF, [1024, 5]]         # 11

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 12
  - [[-1, 7], 1, Concat, [1]]                  # 13
  - [-1, 3, C2f, [512, False]]                 # 14
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 15
  - [[-1, 5], 1, Concat, [1]]                  # 16
  - [-1, 3, C2f, [256, False]]                 # 17
  - [-1, 1, GhostConv, [256, 3, 2]]            # 18
  - [[-1, 14], 1, Concat, [1]]                 # 19
  - [-1, 3, C2f, [512, False]]                 # 20
  - [-1, 1, GhostConv, [512, 3, 2]]            # 21
  - [[-1, 11], 1, Concat, [1]]                 # 22
  - [-1, 3, C2f, [1024, False]]                # 23
  - [[17, 20, 23], 1, Detect, [nc]]            # 24
"""

if __name__ == "__main__":
    import os
    # 重新准备数据（防止重启后丢失）
    if not os.path.exists('/kaggle/working/data.yaml'):
        import zipfile
        zip_path = '/kaggle/input/coco128-jin/coco128.zip' 
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, 'r') as z: z.extractall('/kaggle/working/')
        yaml_content = """path: /kaggle/working/coco128
train: images/train2017
val: images/train2017
nc: 80
names: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']"""
        with open('/kaggle/working/data.yaml', 'w') as f: f.write(yaml_content)

    with open("ghost_cbam_yolov8_true_n.yaml","w",encoding="utf-8") as f:
        f.write(model_yaml)

    print("🚀 正在构建真正的轻量化 Ghost + CBAM 改进模型...")
    # 直接加载 YAML，此时 YOLO 会自动去 tasks.CBAM 里找
    model = YOLO("ghost_cbam_yolov8_true_n.yaml")

    print("开始训练改进模型...")
    result = model.train(
        data="/kaggle/working/data.yaml",  
        epochs=30,                         
        imgsz=320,                         
        batch=8,
        device=0,
        project='/kaggle/working/runs', 
        name='improved_final'
    )
    
    os.system('zip -r /kaggle/working/improved_final.zip /kaggle/working/runs/improved_final')
    print("🎉 改进模型训练完成！请去右侧下载 improved_final.zip！")

In [1]:
import torch
import torch.nn as nn
from ultralytics import YOLO
import ultralytics.nn.tasks as tasks
import os
import zipfile

# ================= 1. 定义模块 =================
from ultralytics.nn.modules import GhostConv

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
    def forward(self, x):
        return torch.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_val, _ = torch.max(x, dim=1, keepdim=True)
        return torch.sigmoid(self.conv(torch.cat([avg, max_val], dim=1)))

class CBAM(nn.Module):
    def __init__(self, c1, ratio=16):
        super().__init__()
        self.ca = ChannelAttention(c1, ratio)
        self.sa = SpatialAttention()
    def forward(self, x):
        x = self.ca(x) * x
        return self.sa(x) * x

# ================= 2. 注入 tasks.py =================
tasks.CBAM = CBAM
tasks.GhostConv = GhostConv

# ================= 3. 纯正的 YOLOv8n 宽度 YAML (最高通道仅256) =================
model_yaml = """
nc: 80
names: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

backbone:
  - [-1, 1, GhostConv, [16, 3, 2]]   # 0-P1/2
  - [-1, 1, GhostConv, [32, 3, 2]]   # 1-P2/4
  - [-1, 1, C2f, [32, True]]         # 2
  - [-1, 1, GhostConv, [64, 3, 2]]   # 3-P3/8
  - [-1, 2, C2f, [64, True]]         # 4
  - [-1, 1, CBAM, [64]]              # 5 (插入CBAM)
  - [-1, 1, GhostConv, [128, 3, 2]]  # 6-P4/16
  - [-1, 2, C2f, [128, True]]        # 7
  - [-1, 1, CBAM, [128]]             # 8 (插入CBAM)
  - [-1, 1, GhostConv, [256, 3, 2]]  # 9-P5/32
  - [-1, 1, C2f, [256, True]]        # 10
  - [-1, 1, SPPF, [256, 5]]          # 11

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 12
  - [[-1, 7], 1, Concat, [1]]                  # 13
  - [-1, 1, C2f, [128, False]]                 # 14
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']] # 15
  - [[-1, 4], 1, Concat, [1]]                  # 16
  - [-1, 1, C2f, [64, False]]                  # 17
  - [-1, 1, GhostConv, [64, 3, 2]]             # 18
  - [[-1, 14], 1, Concat, [1]]                 # 19
  - [-1, 1, C2f, [128, False]]                 # 20
  - [-1, 1, GhostConv, [128, 3, 2]]            # 21
  - [[-1, 11], 1, Concat, [1]]                 # 22
  - [-1, 1, C2f, [256, False]]                 # 23
  - [[17, 20, 23], 1, Detect, [nc]]            # 24
"""

if __name__ == "__main__":
    if not os.path.exists('/kaggle/working/data.yaml'):
        zip_path = '/kaggle/input/coco128-jin/coco128.zip' 
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, 'r') as z: z.extractall('/kaggle/working/')
        yaml_content = """path: /kaggle/working/coco128\ntrain: images/train2017\nval: images/train2017\nnc: 80\nnames: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']"""
        with open('/kaggle/working/data.yaml', 'w') as f: f.write(yaml_content)

    with open("ghost_cbam_yolov8_true_n.yaml","w",encoding="utf-8") as f:
        f.write(model_yaml)

    print("🚀 正在构建 100% 纯正轻量化的 Ghost + CBAM 改进模型...")
    model = YOLO("ghost_cbam_yolov8_true_n.yaml")

    print("开始训练改进模型...")
    result = model.train(
        data="/kaggle/working/data.yaml",  
        epochs=30,                         
        imgsz=320,                         
        batch=8,
        device=0,
        project='/kaggle/working/runs', 
        name='improved_light'
    )
    
    os.system('zip -r /kaggle/working/improved_light.zip /kaggle/working/runs/improved_light')
    print("🎉 训练完成！请去右侧下载 improved_light.zip！")

🚀 正在构建 100% 纯正轻量化的 Ghost + CBAM 改进模型...
开始训练改进模型...
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=ghost_cbam_yolov8_true_n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30     0.387G      4.063      5.784      4.269         82        320: 100% ━━━━━━━━━━━━ 16/16 4.6it/s 3.5s<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 4.9it/s 1.6s0.2ss
                   all        128        929          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30     0.387G      3.983      5.759      4.263        126        320: 6% ╸─────────── 1/16 1.9it/s 0.2s<8.0s

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30     0.387G      3.957      5.753      4.239        102        320: 100% ━━━━━━━━━━━━ 16/16 13.3it/s 1.2s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 18.7it/s 0.4s0.1s
                   all        128        929          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30     0.387G      3.819      5.786      4.222        101        320: 6% ╸─────────── 1/16 2.1it/s 0.1s<7.0s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


... (51 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30     0.389G      3.533      5.391      4.181        104        320: 100% ━━━━━━━━━━━━ 16/16 13.5it/s 1.2s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 13.4it/s 0.6s.2s
                   all        128        929    0.00012    0.00382   0.000111   3.11e-05

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      30/30     0.389G      3.166      5.231       4.12         20        320: 6% ╸─────────── 1/16 2.0it/s 0.1s<7.5s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30     0.389G      3.607      5.357      4.182         42        320: 100% ━━━━━━━━━━━━ 16/16 13.7it/s 1.2s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 13.0it/s 0.6s.2s
                   all        128        929   0.000813    0.00616   0.000175   3.51e-05

30 epochs completed in 0.018 hours.
Optimizer stripped from /kaggle/working/runs/improved_light/weights/last.pt, 6.0MB
Optimizer stripped from /kaggle/working/runs/improved_light/weights/best.pt, 6.0MB

Validating /kaggle/working/runs/improved_light/weights/best.pt...
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
ghost_cbam_YOLOv8_true_n summary (fused): 91 layers, 2,875,044 parameters, 0 gradients, 8.2 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 4.8it/s 1.7s0.2s
                   all        128        929      0.535   0.000333   0.000195  